## `Task` Select best features from from Wine Quality dataset.

* Data Link : https://www.kaggle.com/datasets/rajyellow46/wine-quality

* Drive link : https://docs.google.com/spreadsheets/d/e/2PACX-1vQDVwxneOKOaJL13QMhkAhYrgWlH1tICY7RacUnj_lL8m9uUWaaUf3p7bScNyh_D2Rvt7nc1q11adSy/pub?gid=647503637&single=true&output=csv

Note : Follow approach from Feature Selection Session - 2

In [2]:
# Your Code goes Here
import pandas as pd

data= pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQDVwxneOKOaJL13QMhkAhYrgWlH1tICY7RacUnj_lL8m9uUWaaUf3p7bScNyh_D2Rvt7nc1q11adSy/pub?gid=647503637&single=true&output=csv")

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   type                  6497 non-null   object 
 1   fixed acidity         6487 non-null   float64
 2   volatile acidity      6489 non-null   float64
 3   citric acid           6494 non-null   float64
 4   residual sugar        6495 non-null   float64
 5   chlorides             6495 non-null   float64
 6   free sulfur dioxide   6497 non-null   float64
 7   total sulfur dioxide  6497 non-null   float64
 8   density               6497 non-null   float64
 9   pH                    6488 non-null   float64
 10  sulphates             6493 non-null   float64
 11  alcohol               6497 non-null   float64
 12  quality               6497 non-null   int64  
dtypes: float64(11), int64(1), object(1)
memory usage: 660.0+ KB


In [3]:
data.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [8]:
# Fill missing values with median

data = data.fillna(data.median(numeric_only=True))

In [5]:
data.shape

(6497, 13)

In [7]:
# Percentage of missing values in each column

missing_percent = (data.isnull().mean() * 100).sort_values(ascending=False)

print(missing_percent)

fixed acidity           0.153917
pH                      0.138525
volatile acidity        0.123134
sulphates               0.061567
citric acid             0.046175
chlorides               0.030783
residual sugar          0.030783
type                    0.000000
free sulfur dioxide     0.000000
density                 0.000000
total sulfur dioxide    0.000000
alcohol                 0.000000
quality                 0.000000
dtype: float64


In [10]:
# Find duplicate columns

duplicate_cols = data.columns[data.T.duplicated()]

print("Duplicate Columns:")
print(duplicate_cols)

print("\nTotal Duplicate Columns:", len(duplicate_cols))

Duplicate Columns:
Index([], dtype='object')

Total Duplicate Columns: 0


==>Check

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Encode categorical column

le = LabelEncoder()

data['type'] = le.fit_transform(data['type'])

# Features and target

X = data.drop('quality', axis=1)
y = data['quality']

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Logistic Regression

log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train, y_train)

# Predictions

y_pred = log_reg.predict(X_test)

# Accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.5353846153846153


1.EFS

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from mlxtend.feature_selection import ExhaustiveFeatureSelector

# Encode categorical column

le = LabelEncoder()

data['type'] = le.fit_transform(data['type'])

# Separate features and target

X = data.drop('quality', axis=1)
y = data['quality']

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to DataFrame

X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model

lr = LogisticRegression(max_iter=1000)

# Exhaustive Feature Selector

efs = ExhaustiveFeatureSelector(
    estimator=lr,
    min_features=1,
    max_features=3,
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# Fit EFS

efs.fit(X_train, y_train)

# Best selected features

selected_features = list(efs.best_feature_names_)

print("Selected Features:")
print(selected_features)

# Reduced datasets

X_train_efs = X_train[selected_features]
X_test_efs = X_test[selected_features]

# Train model

lr.fit(X_train_efs, y_train)

# Predictions

y_pred = lr.predict(X_test_efs)

# Accuracy

accuracy = accuracy_score(y_test, y_pred)

print("EFS Accuracy:", accuracy)

Features: 298/298

Selected Features:
['volatile acidity', 'density', 'alcohol']
EFS Accuracy: 0.5407692307692308


2.SBE

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SequentialFeatureSelector

# Encode categorical column

le = LabelEncoder()

data['type'] = le.fit_transform(data['type'])

# Separate features and target

X = data.drop('quality', axis=1)
y = data['quality']

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to DataFrame

X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model

lr = LogisticRegression(max_iter=1000)

# Sequential Backward Selection

sbs = SequentialFeatureSelector(
    estimator=lr,
    n_features_to_select=3,
    direction='backward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# Fit SBS

sbs.fit(X_train, y_train)

# Selected features

selected_features = X_train.columns[sbs.get_support()]

print("Selected Features:")
print(selected_features)

# Reduced datasets

X_train_sbs = X_train[selected_features]
X_test_sbs = X_test[selected_features]

# Train model

lr.fit(X_train_sbs, y_train)

# Predictions

y_pred = lr.predict(X_test_sbs)

# Accuracy

accuracy = accuracy_score(y_test, y_pred)

print("SBS Accuracy:", accuracy)

Selected Features:
Index(['volatile acidity', 'residual sugar', 'alcohol'], dtype='object')
SBS Accuracy: 0.54


3.SFS

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SequentialFeatureSelector

# Encode categorical column

le = LabelEncoder()

data['type'] = le.fit_transform(data['type'])

# Separate features and target

X = data.drop('quality', axis=1)
y = data['quality']

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to DataFrame

X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model

lr = LogisticRegression(max_iter=1000)

# Sequential Forward Selection

sfs = SequentialFeatureSelector(
    estimator=lr,
    n_features_to_select=3,
    direction='forward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# Fit SFS

sfs.fit(X_train, y_train)

# Selected features

selected_features = X_train.columns[sfs.get_support()]

print("Selected Features:")
print(selected_features)

# Reduced datasets

X_train_sfs = X_train[selected_features]
X_test_sfs = X_test[selected_features]

# Train model

lr.fit(X_train_sfs, y_train)

# Predictions

y_pred = lr.predict(X_test_sfs)

# Accuracy

accuracy = accuracy_score(y_test, y_pred)

print("SFS Accuracy:", accuracy)

Selected Features:
Index(['volatile acidity', 'density', 'alcohol'], dtype='object')
SFS Accuracy: 0.5407692307692308


In [ ]:
# ==========================================
# WRAPPER FEATURE SELECTION OBSERVATIONS
# Wine Quality Dataset
# ==========================================

# ==========================================
# 1. Exhaustive Feature Selection (EFS)
# ==========================================

# Selected Features:
# - volatile acidity
# - density
# - alcohol

# Accuracy:
# 54.07%

# Observation:
# EFS checks every possible feature combination
# and selects the best subset globally.

# Since only 3 features were selected,
# these features contribute most strongly
# toward wine quality prediction.

# ==========================================
# 2. Sequential Backward Selection (SBS)
# ==========================================

# Selected Features:
# - volatile acidity
# - residual sugar
# - alcohol

# Accuracy:
# 54%

# Observation:
# SBS started with all features
# and gradually removed least useful features.

# Final selected features were slightly different
# from EFS but accuracy remained almost same.

# ==========================================
# 3. Sequential Forward Selection (SFS)
# ==========================================

# Selected Features:
# - volatile acidity
# - density
# - alcohol

# Accuracy:
# 54.07%

# Observation:
# SFS started with zero features
# and added most useful features step by step.

# Interestingly:
# SFS selected exactly same features as EFS.

# ==========================================
# COMMON OBSERVATIONS
# ==========================================

# Important features repeatedly selected:
# - volatile acidity
# - alcohol

# This indicates:
# These features are highly influential
# in determining wine quality.

# ==========================================
# INTERPRETATION OF FEATURES
# ==========================================

# volatile acidity:
# Higher volatile acidity generally lowers wine quality.

# alcohol:
# Higher alcohol content is often associated
# with better quality wines.

# density:
# Related to sugar and alcohol concentration.

# residual sugar:
# Remaining sugar after fermentation.

# ==========================================
# IMPORTANT LEARNING
# ==========================================

# Even after reducing features to only 3,
# model accuracy remained nearly same (~54%).

# This suggests:
# Many original features were less informative.

# ==========================================
# COMPARISON OF METHODS
# ==========================================

# EFS:
# - Best but computationally expensive
# - Finds globally optimal subset

# SFS:
# - Faster
# - Greedy addition approach

# SBS:
# - Starts with all features
# - Removes least useful features iteratively

# ==========================================
# FINAL CONCLUSION
# ==========================================

# Wrapper methods successfully reduced
# dimensionality while maintaining similar accuracy.

# Only a small subset of features was sufficient
# for predicting wine quality reasonably well.